In [19]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# reading the file
df = pd.read_csv("Housing.csv")
df_encoded = pd.get_dummies(df, columns=["mainroad", "guestroom", "basement", "hotwaterheating", "airconditioning", "prefarea", "furnishingstatus"])

# Starting from your df_encoded
X = df_encoded.drop(["price"], axis=1).copy()
y = df_encoded["price"].copy()

# Preprocessing for price and area , where we standardize their values 
mean_area = X["area"].mean()
std_area = X["area"].std()
mean_price = y.mean()
std_price = y.std()

X["area"] = (X["area"] - mean_area) / std_area
y = (y - mean_price) / std_price

# 80/20 split for both models
X_train_df, X_test_df, y_train_s, y_test_s = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train_df:", X_train_df.shape, "X_test_df:", X_test_df.shape)
print("y_train_s:", y_train_s.shape, "y_test_s:", y_test_s.shape)

X_train_df: (436, 20) X_test_df: (109, 20)
y_train_s: (436,) y_test_s: (109,)


In [21]:
#Comparing Neural Network to the basic Linear Regression Model from sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# sklearn expects (m, n_features)
lr = LinearRegression()
lr.fit(X_train_df, y_train_s)

# Predictions (scaled target units, same as your NN setup)
lr_pred_test = lr.predict(X_test_df)
lr_pred_train = lr.predict(X_train_df)

# Metrics in scaled units
lr_mse_train = mean_squared_error(y_train_s, lr_pred_train)
lr_mse_test = mean_squared_error(y_test_s, lr_pred_test)
lr_mae_test = mean_absolute_error(y_test_s, lr_pred_test)
lr_r2_test = r2_score(y_test_s, lr_pred_test)

print("=== sklearn LinearRegression (scaled target) ===")
print(f"Train MSE: {lr_mse_train:.6f}")
print(f"Test  MSE: {lr_mse_test:.6f}")
print(f"Test  MAE: {lr_mae_test:.6f}")
print(f"Test   R²: {lr_r2_test:.6f}")

=== sklearn LinearRegression (scaled target) ===
Train MSE: 0.276789
Test  MSE: 0.501442
Test  MAE: 0.518618
Test   R²: 0.652924


In [12]:
from nn_constructor import NeuralNetwork, Layer, LinearAct, ReluAct, MeanRegressionError

# Convert to your NN format: (features, m)
X_train_nn = np.asarray(X_train_df.T, dtype=float)
X_test_nn  = np.asarray(X_test_df.T, dtype=float)
y_train_nn = np.asarray(y_train_s, dtype=float).reshape(1, -1)
y_test_nn  = np.asarray(y_test_s, dtype=float).reshape(1, -1)

# Build a fresh network (20 -> 8 -> 1)
layer1 = Layer(in_features=X_train_nn.shape[0], out_features= 20, activation=ReluAct())
layer2 = Layer(in_features=20, out_features=8, activation=ReluAct())
layer3 = Layer(in_features=8, out_features=1, activation=LinearAct())

layers = [layer1, layer2, layer3]

loss = MeanRegressionError(m=X_train_nn.shape[1], y=y_train_nn, lamda=0.001)
nn = NeuralNetwork(alpha=0.02, data=X_train_nn, loss=loss, layers=layers)

nn.fit()

# Raw predictions (regression => use raw outputs)
nn_pred_train = nn.predict(X_train_nn)
nn_pred_test = nn.predict(X_test_nn)

# Your own MSE function (if it returns value)
print("=== Your NN (scaled target) ===")
print("Train MSE (nn.meansquarederror):", nn.meansquarederror(nn_pred_train, y_train_nn))
print("Test  MSE (nn.meansquarederror):", nn.meansquarederror(nn_pred_test, y_test_nn))

epoch 1 - data_loss(MSE): 1.203594 | reg_loss: 0.000060 | total: 1.203654
epoch 2 - data_loss(MSE): 1.129733 | reg_loss: 0.000060 | total: 1.129794
epoch 3 - data_loss(MSE): 1.088135 | reg_loss: 0.000060 | total: 1.088195
epoch 4 - data_loss(MSE): 1.061948 | reg_loss: 0.000060 | total: 1.062008
epoch 5 - data_loss(MSE): 1.043445 | reg_loss: 0.000060 | total: 1.043505
epoch 6 - data_loss(MSE): 1.029030 | reg_loss: 0.000060 | total: 1.029091
epoch 7 - data_loss(MSE): 1.016997 | reg_loss: 0.000060 | total: 1.017057
epoch 8 - data_loss(MSE): 1.006423 | reg_loss: 0.000060 | total: 1.006483
epoch 9 - data_loss(MSE): 0.996834 | reg_loss: 0.000060 | total: 0.996894
epoch 10 - data_loss(MSE): 0.987935 | reg_loss: 0.000060 | total: 0.987995
epoch 11 - data_loss(MSE): 0.979582 | reg_loss: 0.000060 | total: 0.979642
epoch 12 - data_loss(MSE): 0.971686 | reg_loss: 0.000060 | total: 0.971746
epoch 13 - data_loss(MSE): 0.964163 | reg_loss: 0.000060 | total: 0.964224
epoch 14 - data_loss(MSE): 0.95698